# Shazam-Style MCID Audio Matching POC

This notebook demonstrates a small proof of concept for matching short claim audio samples against a reference library of JFP audio and suggesting MCID candidates.

## 1. Purpose

Given a short audio sample from a YouTube claim video, create an audio fingerprint, compare it against known JFP reference audio, and return the top MCID matches.

## 2. Difference from Bryce

Bryce is working on audio-to-text or language detection. This prototype is audio-to-reference matching, like Shazam, where the output is a suggested MCID, not a transcript.

## 3. Inputs

- `data/reference_library.csv`: approved/public reference audio mapped to MCIDs
- `data/claim_samples.csv`: short claim samples to test
- `data/reference_audio/`: reference WAV/MP3/M4A files
- `data/claim_audio/`: claim sample WAV/MP3/M4A files

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from fingerprint_matcher import (
    build_reference_index,
    confidence_from_results,
    load_reference_library,
    match_audio_sample,
)

PROJECT_ROOT

## 4. Load Claim and Reference Data

In [ ]:
reference_library_path = PROJECT_ROOT / 'data' / 'reference_library.csv'
claim_samples_path = PROJECT_ROOT / 'data' / 'claim_samples.csv'

reference_df = pd.read_csv(reference_library_path)
claim_df = pd.read_csv(claim_samples_path)

display(reference_df)
display(claim_df)

## 5. Build Reference Fingerprint Library

In [ ]:
tracks = load_reference_library(reference_library_path)
index, track_by_id = build_reference_index(tracks)

len(tracks), len(index)

## 6. Match Claim Audio Samples

In [ ]:
rows = []

for claim in claim_df.fillna('').to_dict('records'):
    claim_audio_file = PROJECT_ROOT / claim['claim_audio_file']
    results = match_audio_sample(claim_audio_file, index, track_by_id, top_n=3)
    confidence, action = confidence_from_results(results)

    row = {
        'source_file': claim.get('source_file', ''),
        'claim_row': claim.get('claim_row', ''),
        'video_title': claim.get('video_title', ''),
        'video_duration_sec': claim.get('video_duration_sec', ''),
        'existing_mcid': claim.get('existing_mcid', ''),
        'confidence': confidence,
        'recommended_action': action,
    }

    for i in range(3):
        result = results[i] if i < len(results) else None
        n = i + 1
        row[f'suggested_mcid_{n}'] = result.mcid if result else ''
        row[f'match_title_{n}'] = result.title if result else ''
        row[f'audio_score_{n}'] = result.audio_score if result else ''
        row[f'matched_timestamp_{n}'] = result.matched_timestamp_sec if result else ''

    rows.append(row)

results_df = pd.DataFrame(rows)
display(results_df)

## 7. Export Results

In [ ]:
output_path = PROJECT_ROOT / 'outputs' / 'mcid_audio_match_results.xlsx'
results_df.to_excel(output_path, index=False)
output_path

## 8. Graphs

In [ ]:
if not results_df.empty:
    results_df['confidence'].value_counts().plot(kind='bar', title='Strong / Possible / Weak Match Counts')
    plt.xlabel('Confidence')
    plt.ylabel('Claim samples')
    plt.show()

    pd.to_numeric(results_df['audio_score_1'], errors='coerce').plot(kind='hist', bins=10, title='Top Audio Score Distribution')
    plt.xlabel('Audio score')
    plt.show()

## 9. Limitations

- This starter fingerprint is for proof-of-concept only.
- Music-heavy sections may confuse matches across language versions.
- Public YouTube audio may be compressed, edited, or different from official source audio.
- A stronger production version should use a dedicated fingerprint engine and an approved internal reference library.

## 10. Next Steps

1. Test with 3 to 5 public reference files.
2. Add 5 to 10 claim samples.
3. Compare audio match results against known MCIDs.
4. Show the results to Ken before requesting internal JFP audio access.
5. Convert the working notebook into a Streamlit demo.